# AI Cup 2026 — Bird Radar Track Classification
Two-stage classifier: binary Gull detector + ensembled 8-class non-Gull classifier.
Generates `submission.csv`.

## Model Hyperparameters

In [ ]:
# ─── Model Hyperparameters ───
N_ESTIMATORS      = 1000
LEARNING_RATE     = 0.05
NUM_LEAVES        = 63
MIN_CHILD_SAMPLES = 10
SUBSAMPLE         = 0.8
COLSAMPLE_BYTREE  = 0.8
CLASS_WEIGHT      = 'balanced'
DEVICE            = 'gpu'

# ─── CV Settings ───
N_SPLITS          = 10
RANDOM_STATE      = 42

# ─── Weak-class boost (row duplication) ───
CLASS_BOOST = {'Cormorants': 3, 'Waders': 3, 'Geese': 3}

## Imports

In [ ]:
import numpy as np
import pandas as pd
from shapely import wkb
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from lightgbm import LGBMClassifier
import lightgbm as lgb
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC
from catboost import CatBoostClassifier

## 1. Load Data (OpenMeteo-enriched)

In [ ]:
print("Loading OpenMeteo-enriched datasets...")
train_df = pd.read_csv("dataset/train_with_openmeteo.csv").set_index("track_id")
test_df = pd.read_csv("dataset/test_with_openmeteo.csv").set_index("track_id")
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

# Wind config for OpenMeteo
WIND_SPEED_COL = 'openmeteo_wind_speed_10m_kmh'
WIND_SPEED_OBS_COL = 'openmeteo_wind_speed_10m_kmh'
WIND_DIR_COL = 'openmeteo_wind_direction_10m_degrees'
WIND_UNIT_FACTOR = 1 / 3.6  # km/h -> m/s

## 2. Trajectory Parsing & Feature Extraction

In [ ]:
def parse_trajectory(hex_str):
    """Decode EWKB hex string into a list of (lon, lat, alt, rcs) tuples."""
    if not isinstance(hex_str, str) or len(hex_str) == 0:
        return []
    try:
        geom = wkb.loads(bytes.fromhex(hex_str) if not hex_str.startswith('\x01') else hex_str, hex=True)
        if geom.geom_type == 'LineString':
            return list(geom.coords)
        elif geom.geom_type == 'Point':
            return [geom.coords[0]]
        elif geom.geom_type in ('MultiPoint', 'GeometryCollection'):
            return [g.coords[0] for g in geom.geoms]
    except Exception:
        pass
    return []


def trajectory_features(row):
    """Extract spatial, RCS, and velocity features from a trajectory."""
    coords = parse_trajectory(row['trajectory'])
    n = len(coords)
    if n == 0:
        return pd.Series({})

    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    alts = [c[2] for c in coords] if len(coords[0]) > 2 else [np.nan] * n
    rcs  = [c[3] for c in coords] if len(coords[0]) > 3 else [np.nan] * n

    # Trajectory smoothing (rolling window=3) to reduce radar noise
    lons_s = pd.Series(lons).rolling(window=3, min_periods=1, center=True).mean().values
    lats_s = pd.Series(lats).rolling(window=3, min_periods=1, center=True).mean().values

    # Horizontal displacement (degrees -> approx metres at ~53 deg N)
    dx = np.diff(lons_s) * 71000
    dy = np.diff(lats_s) * 111000
    step_dist = np.sqrt(dx**2 + dy**2)
    total_dist = step_dist.sum() if len(step_dist) > 0 else 0.0

    # Tortuosity (turning behaviour)
    bearings = np.arctan2(dy, dx)
    bearing_changes = np.abs(np.diff(bearings)) if len(bearings) > 1 else np.array([0.0])
    bearing_changes = np.minimum(bearing_changes, 2 * np.pi - bearing_changes)

    # Sharp turn ratio: fraction of turns > 45 degrees
    sharp_turn_ratio = (bearing_changes > np.pi / 4).mean() if len(bearing_changes) > 0 else 0.0

    # Straightness index: displacement / total path length
    displacement = np.sqrt(
        ((lons[-1] - lons[0]) * 71000) ** 2 +
        ((lats[-1] - lats[0]) * 111000) ** 2
    )
    straightness = displacement / (total_dist + 1e-6) if total_dist > 0 else 0.0

    # Sinuosity: total path / displacement
    sinuosity = total_dist / (displacement + 1e-6)

    # Track heading (overall direction from start to end)
    track_heading_rad = np.arctan2(
        (lats[-1] - lats[0]) * 111000,
        (lons[-1] - lons[0]) * 71000
    )

    # Altitude change features
    alt_arr = np.array(alts, dtype=float)
    if n > 1 and not np.all(np.isnan(alt_arr)):
        alt_changes = np.diff(alt_arr)
        alt_climb_rate = np.nanmean(alt_changes)
        alt_descent_rate = np.nanmin(alt_changes)
        alt_variability = np.nanstd(alt_changes)
    else:
        alt_climb_rate = 0.0
        alt_descent_rate = 0.0
        alt_variability = 0.0

    # RCS range
    rcs_range = np.nanmax(rcs) - np.nanmin(rcs)

    feats = {
        'n_points':       n,
        'total_dist_m':   total_dist,
        'mean_step_m':    step_dist.mean() if len(step_dist) > 0 else 0.0,
        'std_step_m':     step_dist.std()  if len(step_dist) > 0 else 0.0,
        'lon_range':      max(lons) - min(lons),
        'lat_range':      max(lats) - min(lats),
        'alt_mean':       np.nanmean(alts),
        'alt_std':        np.nanstd(alts),
        'rcs_mean':       np.nanmean(rcs),
        'rcs_std':        np.nanstd(rcs),
        'rcs_min':        np.nanmin(rcs),
        'rcs_max':        np.nanmax(rcs),
        'rcs_range':      rcs_range,
        'tortuosity':     bearing_changes.mean() if len(bearing_changes) > 0 else 0.0,
        'tortuosity_max': bearing_changes.max()  if len(bearing_changes) > 0 else 0.0,
        'sharp_turn_ratio':   sharp_turn_ratio,
        'straightness':       straightness,
        'sinuosity':          sinuosity,
        'lon_mean':           np.mean(lons),
        'lat_mean':           np.mean(lats),
        'track_heading_rad':  track_heading_rad,
        'alt_climb_rate':     alt_climb_rate,
        'alt_descent_rate':   alt_descent_rate,
        'alt_variability':    alt_variability,
    }

    # Speed & acceleration from trajectory_time
    times = row.get('trajectory_time', '')
    t_list = []
    if isinstance(times, str) and times.strip():
        try:
            t_list = [float(x) for x in times.strip('[]').split(',')]
        except ValueError:
            pass
    elif isinstance(times, (list, np.ndarray)):
        t_list = list(times)

    if len(t_list) == n and n > 1:
        dt = np.diff(t_list)
        dt = np.where(dt == 0, 1e-6, dt)
        speeds = step_dist / dt
        feats['speed_mean'] = np.mean(speeds)
        feats['speed_std']  = np.std(speeds)
        feats['speed_max']  = np.max(speeds)
        feats['speed_cv']   = np.std(speeds) / (np.mean(speeds) + 1e-6)

        if len(speeds) > 1:
            accel = np.diff(speeds) / dt[1:]
            feats['accel_mean'] = np.mean(accel)
            feats['accel_std']  = np.std(accel)
        else:
            feats['accel_mean'] = 0.0
            feats['accel_std']  = 0.0
    else:
        feats['speed_mean'] = np.nan
        feats['speed_std']  = np.nan
        feats['speed_max']  = np.nan
        feats['speed_cv']   = np.nan
        feats['accel_mean'] = np.nan
        feats['accel_std']  = np.nan

    return pd.Series(feats)

## 3. Feature Engineering

In [ ]:
for df in [train_df, test_df]:
    # Timestamps
    df['ts_start'] = pd.to_datetime(df['timestamp_start_radar_utc'], utc=True)
    df['ts_end']   = pd.to_datetime(df['timestamp_end_radar_utc'],   utc=True)
    df['duration_s'] = (df['ts_end'] - df['ts_start']).dt.total_seconds()

    # Time-of-day / seasonality
    df['hour']  = df['ts_start'].dt.hour
    df['month'] = df['ts_start'].dt.month
    df['is_daytime'] = ((df['hour'] >= 6) & (df['hour'] <= 20)).astype(int)

    # Cyclical encoding
    df['hour_sin']  = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # Derived radar features
    df['alt_range']      = df['max_z'] - df['min_z']
    df['airspeed_per_m'] = df['airspeed'] / (df['max_z'] + 1)

    # Wind-relative features (convert to m/s)
    wf = WIND_UNIT_FACTOR
    df['headwind_component'] = df['airspeed'] - df[WIND_SPEED_OBS_COL] * wf
    df['airspeed_wind_ratio'] = df['airspeed'] / (df[WIND_SPEED_COL] * wf + 0.1)

    # Wind direction sin/cos
    wd = df[WIND_DIR_COL]
    df['openmeteo_wind_dir_sin'] = np.sin(2 * np.pi * wd / 360)
    df['openmeteo_wind_dir_cos'] = np.cos(2 * np.pi * wd / 360)

# Trajectory features
print("Extracting trajectory features for train_df...")
train_df = train_df.join(train_df.apply(trajectory_features, axis=1))
print("Extracting trajectory features for test_df...")
test_df = test_df.join(test_df.apply(trajectory_features, axis=1))

# Post-trajectory interaction features
for df in [train_df, test_df]:
    wf = WIND_UNIT_FACTOR
    df['rcs_speed_ratio'] = df['rcs_mean'] / (df['airspeed'] + 1e-6)

    # Altitude-adjusted wind speed (Hellmann power law, exponent 0.143)
    df['alt_adjusted_wind_speed'] = (
        df[WIND_SPEED_COL] * wf
        * np.power(np.maximum(df['alt_mean'], 10) / 10.0, 0.143)
    )

    # True tailwind & crosswind from track heading vs wind direction
    wind_dir_rad = np.deg2rad(df[WIND_DIR_COL])
    heading = df['track_heading_rad']
    angle_diff = wind_dir_rad - heading
    df['true_tailwind_component'] = df['alt_adjusted_wind_speed'] * np.cos(angle_diff)
    df['true_crosswind_component'] = np.abs(df['alt_adjusted_wind_speed'] * np.sin(angle_diff))

## 4. Feature List

In [ ]:
base_features = [
    'airspeed', 'min_z', 'max_z', 'duration_s', 'radar_bird_size',
    'hour', 'month', 'is_daytime',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'alt_range', 'airspeed_per_m',
    'headwind_component', 'airspeed_wind_ratio',
    'rcs_speed_ratio', 'alt_adjusted_wind_speed',
    'true_tailwind_component', 'true_crosswind_component',
]

trajectory_feats = [
    'n_points', 'total_dist_m', 'mean_step_m', 'std_step_m',
    'lon_range', 'lat_range', 'alt_mean', 'alt_std',
    'rcs_mean', 'rcs_std', 'rcs_min', 'rcs_max', 'rcs_range',
    'tortuosity', 'tortuosity_max', 'sharp_turn_ratio',
    'straightness', 'sinuosity',
    'lon_mean', 'lat_mean', 'track_heading_rad',
    'alt_climb_rate', 'alt_descent_rate', 'alt_variability',
    'speed_mean', 'speed_std', 'speed_max', 'speed_cv',
    'accel_mean', 'accel_std',
]

openmeteo_features = [
    'openmeteo_air_temperature_2m_c',
    'openmeteo_relative_humidity_2m_percent',
    'openmeteo_dew_point_2m_c',
    'openmeteo_precipitation_mm',
    'openmeteo_cloud_cover_percent',
    'openmeteo_pressure_msl_hpa',
    'openmeteo_weather_code',
    'openmeteo_wind_speed_10m_kmh',
    'openmeteo_wind_direction_10m_degrees',
    'openmeteo_wind_gusts_10m_kmh',
    'openmeteo_shortwave_radiation_w_m2',
    'openmeteo_sunshine_duration_s',
    'openmeteo_vapour_pressure_deficit_kpa',
    'openmeteo_is_day',
    'openmeteo_wind_dir_sin',
    'openmeteo_wind_dir_cos',
]

features = base_features + trajectory_feats + openmeteo_features

X = train_df[features]
X_test = test_df[features]
y = train_df['bird_group']

print(f"Feature matrix: X={X.shape}, X_test={X_test.shape}, classes={y.nunique()}")

## 5. Model & Pipeline (Two-Stage)

In [ ]:
numeric_features = [f for f in features if f != 'radar_bird_size']
categorical_features = ['radar_bird_size']
cat_indices = [len(numeric_features) + i for i in range(len(categorical_features))]

imputer = ColumnTransformer([
    ('num_imputer', SimpleImputer(strategy='median'), numeric_features),
    ('cat_imputer', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), categorical_features)
])

oversampler = SMOTENC(categorical_features=cat_indices, random_state=RANDOM_STATE)

lgb_params = dict(
    n_estimators=N_ESTIMATORS, learning_rate=LEARNING_RATE,
    num_leaves=NUM_LEAVES, min_child_samples=MIN_CHILD_SAMPLES,
    subsample=SUBSAMPLE, colsample_bytree=COLSAMPLE_BYTREE,
    class_weight=CLASS_WEIGHT, random_state=RANDOM_STATE,
    n_jobs=-1, device=DEVICE, verbose=-1,
)

# CatBoost setup
cb_cat_idx = [len(numeric_features) + i for i in range(len(categorical_features))]
cb_params = dict(
    iterations=1000, learning_rate=0.05, depth=8, l2_leaf_reg=3,
    auto_class_weights='Balanced', random_seed=42, task_type='GPU', verbose=0,
)
cb_imputer = ColumnTransformer([
    ('num_imputer', SimpleImputer(strategy='median'), numeric_features),
    ('cat_imputer', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
    ]), categorical_features)
])

def apply_boost(X_arr, y_arr, class_boost):
    """Duplicate rows for weak classes."""
    if not class_boost:
        return X_arr, y_arr
    X_np = X_arr if isinstance(X_arr, np.ndarray) else np.asarray(X_arr)
    y_np = y_arr if isinstance(y_arr, np.ndarray) else np.asarray(y_arr)
    extra_X, extra_y = [], []
    for cls, mult in class_boost.items():
        repeats = int(mult) - 1
        if repeats <= 0:
            continue
        cls_mask = (y_np == cls)
        X_cls = X_np[cls_mask]
        y_cls = y_np[cls_mask]
        if len(X_cls) > 0:
            extra_X.extend([X_cls] * repeats)
            extra_y.extend([y_cls] * repeats)
    if extra_X:
        X_np = np.vstack([X_np] + extra_X)
        y_np = np.concatenate([y_np] + extra_y)
    return X_np, y_np

print("Stage 1 (Gull binary): LightGBM")
print("Stage 2 (non-Gull 8-class): LightGBM + CatBoost ensemble")

## 6. Two-Stage Cross-Validation + Training

In [ ]:
groups = train_df['primary_observation_id']
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
split = list(cv.split(X, y, groups))

classes = np.sort(y.unique())
needed_columns = [
    "Clutter", "Cormorants", "Pigeons", "Ducks", "Geese",
    "Gulls", "Birds of Prey", "Waders", "Songbirds",
]

oof_preds = pd.DataFrame(0.0, index=X.index, columns=classes)
test_preds = np.zeros((len(X_test), len(classes)))

print(f"Training {N_SPLITS}-fold two-stage pipeline...")
for i, (train_idx, val_idx) in enumerate(split):
    X_tr = X.iloc[train_idx]
    X_va = X.iloc[val_idx]
    y_tr = y.iloc[train_idx]
    y_va = y.iloc[val_idx]

    # ════════════════════════════════════════════
    # STAGE 1: Binary Gull vs Non-Gull (LightGBM)
    # ════════════════════════════════════════════
    y_tr_bin = (y_tr == 'Gulls').astype(int)

    lgb_binary = LGBMClassifier(
        n_estimators=1000, learning_rate=0.05, num_leaves=63,
        min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=42, n_jobs=-1,
        device='gpu', verbose=-1,
    )
    pipe1 = ImbPipeline([
        ('imputer', clone(imputer)),
        ('oversampler', 'passthrough'),
        ('model', lgb_binary)
    ])
    pipe1.fit(X_tr, y_tr_bin)
    val_p_gull = pipe1.predict_proba(X_va)[:, 1]
    test_p_gull = pipe1.predict_proba(X_test)[:, 1]

    # ════════════════════════════════════════════
    # STAGE 2: 8-class Non-Gull (LightGBM + CatBoost ensemble)
    # ════════════════════════════════════════════
    non_gull_mask = y_tr != 'Gulls'
    X_tr_ng = X_tr[non_gull_mask]
    y_tr_ng = y_tr[non_gull_mask]

    # ── Stage 2a: LightGBM with oversampling + boost-weak ──
    lgb_imp = clone(imputer)
    X_tr_ng_imp = lgb_imp.fit_transform(X_tr_ng, y_tr_ng)
    X_va_imp = lgb_imp.transform(X_va)
    X_te_imp = lgb_imp.transform(X_test)

    os_step = clone(oversampler)
    X_ng_res, y_ng_res = os_step.fit_resample(X_tr_ng_imp, y_tr_ng)
    X_ng_res, y_ng_res = apply_boost(X_ng_res, y_ng_res, CLASS_BOOST)

    lgb_s2 = LGBMClassifier(**lgb_params)
    lgb_s2.set_params(min_child_samples=max(10, int(len(X_ng_res) * 0.01)))
    lgb_s2.fit(X_ng_res, y_ng_res)
    lgb_val_proba = lgb_s2.predict_proba(X_va_imp)
    lgb_test_proba = lgb_s2.predict_proba(X_te_imp)
    s2_classes = lgb_s2.classes_

    # ── Stage 2b: CatBoost ──
    cb_imp2 = clone(cb_imputer)
    X_tr_ng_cb = cb_imp2.fit_transform(X_tr_ng, y_tr_ng)
    X_va_cb2 = cb_imp2.transform(X_va)
    X_te_cb2 = cb_imp2.transform(X_test)
    for ci in cb_cat_idx:
        X_tr_ng_cb[:, ci] = X_tr_ng_cb[:, ci].astype(str)
        X_va_cb2[:, ci] = X_va_cb2[:, ci].astype(str)
        X_te_cb2[:, ci] = X_te_cb2[:, ci].astype(str)

    cb_s2 = CatBoostClassifier(**cb_params)
    cb_s2.fit(X_tr_ng_cb, y_tr_ng, cat_features=cb_cat_idx)
    cb_val_proba = cb_s2.predict_proba(X_va_cb2)
    cb_test_proba = cb_s2.predict_proba(X_te_cb2)

    # Align CatBoost classes to LightGBM class order
    cb_s2_classes = cb_s2.classes_
    if not np.array_equal(cb_s2_classes, s2_classes):
        cb_reorder = [list(cb_s2_classes).index(c) for c in s2_classes]
        cb_val_proba = cb_val_proba[:, cb_reorder]
        cb_test_proba = cb_test_proba[:, cb_reorder]

    # ── Stage 2 ensemble: simple average ──
    val_p_rest = (lgb_val_proba + cb_val_proba) / 2
    test_p_rest = (lgb_test_proba + cb_test_proba) / 2

    # ════════════════════════════════════════════
    # COMBINE: p(Gull) from Stage 1, p(other) from Stage 2
    # ════════════════════════════════════════════
    val_combined = np.zeros((len(X_va), len(classes)))
    test_combined = np.zeros((len(X_test), len(classes)))

    gull_idx = list(classes).index('Gulls')
    val_combined[:, gull_idx] = val_p_gull
    test_combined[:, gull_idx] = test_p_gull

    for j, cls in enumerate(s2_classes):
        cls_idx = list(classes).index(cls)
        val_combined[:, cls_idx] = (1 - val_p_gull) * val_p_rest[:, j]
        test_combined[:, cls_idx] = (1 - test_p_gull) * test_p_rest[:, j]

    oof_preds.iloc[val_idx] = val_combined
    test_preds += test_combined

    fold_ap = average_precision_score(
        pd.get_dummies(y_va).reindex(columns=classes, fill_value=0),
        val_combined, average='macro'
    )
    print(f"  Fold {i+1}/{N_SPLITS} — val mAP: {fold_ap:.4f}")

test_preds /= N_SPLITS

## 7. Evaluation

In [ ]:
# Build ground-truth OOF solution
solution_df = (
    train_df
    .reset_index()
    .groupby(["track_id", "bird_group"])
    .size()
    .unstack(fill_value=0)
)

# Align OOF predictions to solution_df
oof_aligned = oof_preds.loc[solution_df.index, solution_df.columns]

overall_map = average_precision_score(
    solution_df[needed_columns],
    oof_aligned[needed_columns],
    average='macro'
)

print(f"{'='*50}")
print(f" OOF Macro-Averaged AP (mAP): {overall_map:.4f}")
print(f"{'='*50}")
print("\n Per-Class Average Precision:")
for cls in needed_columns:
    if cls in solution_df.columns and cls in oof_aligned.columns:
        ap = average_precision_score(solution_df[cls], oof_aligned[cls])
        print(f"   {cls:20s}: {ap:.4f}")

## 8. Generate Submission

In [ ]:
submission_df = pd.DataFrame(
    test_preds,
    index=X_test.index,
    columns=classes
)
submission_df.index.name = 'track_id'
submission_df.to_csv('submission.csv')
print(f"Saved submission.csv ({len(submission_df)} rows)")